In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 設定繁體中文與負號顯示
plt.rcParams['font.sans-serif'] = ['Microsoft JhengHei', 'SimHei', 'Arial']
plt.rcParams['axes.unicode_minus'] = False


In [ ]:
# ==============================================================================
# 1. 工程參數與情境設定
# ==============================================================================
TARGET = 10.00       # 外徑名義目標值 (mm)
TOLERANCE = 0.05     # 公差 ±0.05 mm
USL = TARGET + TOLERANCE  # 規格上限 10.05 mm
LSL = TARGET - TOLERANCE  # 規格下限 9.95 mm

# 製程設計固有能力（共同原因標準差）：設定六標準差剛好落在公差內 (Cp ≈ 1.33)
SIGMA_COMMON = 0.0125  

SUBGROUP_SIZE = 5         # 子組大小 n = 5
SAMPLING_INTERVAL = 2     # 每 2 小時抽一次
TOTAL_HOURS = 40          # 模擬連續生產 40 小時 (相當於 5 個班次)
TOTAL_SUBGROUPS = TOTAL_HOURS // SAMPLING_INTERVAL  # 共 20 個子組

# n = 5 時的 SPC 常數表
A2 = 0.577
D3 = 0.000
D4 = 2.114
d2 = 2.326

In [ ]:
# ==============================================================================
# 2. 模擬生產歷程與合理子組抽樣
# ==============================================================================
np.random.seed(42)  # 固定隨機因子以便重現

subgroup_data = []
timestamps = []

for k in range(1, TOTAL_SUBGROUPS + 1):
    current_hour = k * SAMPLING_INTERVAL
    timestamps.append(f"{current_hour}h")
    
    # 前 24 小時（組 1~12）：製程受控（純共同原因）
    # 第 26~32 小時（組 13~16）：刀具磨損，平均值上漂
    # 第 34~40 小時（組 17~20）：換刀偏差 +0.02 mm
    if k <= 12:
        true_mean = TARGET
    elif 13 <= k <= 16:
        true_mean = TARGET + (k - 12) * 0.008
    else:
        true_mean = TARGET + 0.022
    
    # 合理子組：連續生產抽 5 支，條件一致
    samples = np.random.normal(loc=true_mean, scale=SIGMA_COMMON, size=SUBGROUP_SIZE)
    subgroup_data.append(samples)

df = pd.DataFrame(subgroup_data, columns=[f"樣本{i+1}" for i in range(SUBGROUP_SIZE)])
df['子組編號'] = range(1, TOTAL_SUBGROUPS + 1)
df['生產時間'] = timestamps
df['X_bar'] = df[[f"樣本{i+1}" for i in range(SUBGROUP_SIZE)]].mean(axis=1)
df['R'] = df[[f"樣本{i+1}" for i in range(SUBGROUP_SIZE)]].max(axis=1) - \
          df[[f"樣本{i+1}" for i in range(SUBGROUP_SIZE)]].min(axis=1)

In [ ]:
# ==============================================================================
# 3. 計算管制界限（採用前 12 組受控狀態作為基準基線）
# ==============================================================================
baseline_df = df[df['子組編號'] <= 12]
X_double_bar = baseline_df['X_bar'].mean()
R_bar = baseline_df['R'].mean()

# X-bar 管制界限
UCL_X = X_double_bar + A2 * R_bar
CL_X = X_double_bar
LCL_X = X_double_bar - A2 * R_bar

# R 管制界限
UCL_R = D4 * R_bar
CL_R = R_bar
LCL_R = D3 * R_bar

# 估計組內標準差與製程能力
sigma_within = R_bar / d2
Cp = (USL - LSL) / (6 * sigma_within)
Cpu = (USL - X_double_bar) / (3 * sigma_within)
Cpl = (X_double_bar - LSL) / (3 * sigma_within)
Cpk = min(Cpu, Cpl)

In [ ]:
# ==============================================================================
# 4. 終端輸出分析報告
# ==============================================================================
print("=" * 65)
print("       精密軸心加工 SPC 分析報告（抽樣頻率 2hr，子組 n=5）")
print("=" * 65)
print(f"規格限制：USL = {USL:.3f} mm, LSL = {LSL:.3f} mm, 目標值 = {TARGET:.3f} mm")
print(f"基準期（前 12 組受控狀態）統計量：")
print(f"  - 總平均值 (X-double-bar) : {X_double_bar:.4f} mm")
print(f"  - 平均全距 (R-bar)        : {R_bar:.4f} mm")
print(f"  - 估計組內標準差 (σ_within) : {sigma_within:.4f} mm")
print(f"  - 製程能力指標 Cp / Cpk   : {Cp:.2f} / {Cpk:.2f}")
print("-" * 65)
print("管制界限 (Control Limits)：")
print(f"  - X-bar 圖 : UCL = {UCL_X:.4f} mm, CL = {CL_X:.4f} mm, LCL = {LCL_X:.4f} mm")
print(f"  - R 圖     : UCL = {UCL_R:.4f} mm, CL = {CL_R:.4f} mm, LCL = {LCL_R:.4f} mm")
print("=" * 65)

In [ ]:
# ==============================================================================
# 5. 繪製標準 X-bar & R 管制圖（注意使用 r"" 原生字串防止 \b 跳脫）
# ==============================================================================
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 8), sharex=True)

subgroups = df['子組編號']

# --- 子圖 1：X-bar 管制圖 ---
ax1.plot(subgroups, df['X_bar'], marker='o', color='#1f77b4', linewidth=1.5, label=r'子組平均值 ($\bar{X}$)')
ax1.axhline(UCL_X, color='red', linestyle='--', label=f'UCL = {UCL_X:.3f}')
ax1.axhline(CL_X, color='green', linestyle='-', label=f'CL = {CL_X:.3f}')
ax1.axhline(LCL_X, color='red', linestyle='--', label=f'LCL = {LCL_X:.3f}')

# 標註超出管制界限的點
out_of_control = df[(df['X_bar'] > UCL_X) | (df['X_bar'] < LCL_X)]
ax1.plot(out_of_control['子組編號'], out_of_control['X_bar'], 'ro', markersize=8, label='特殊原因警報 (失控)')

ax1.axvline(12.5, color='gray', linestyle=':')
ax1.text(6, UCL_X + 0.005, '【受控狀態】\n僅有共同原因', ha='center', fontsize=9, color='darkgreen')
ax1.text(14.5, UCL_X + 0.005, '【刀具磨損】\n平均值漂移', ha='center', fontsize=9, color='darkorange')
ax1.text(18.5, UCL_X + 0.005, '【換刀偏差】\n中心偏高', ha='center', fontsize=9, color='darkred')

# 使用 r"" 避免 \b 被當成 backspace
ax1.set_title(r"$\bar{X}$ 管制圖（反映「子組間變異」：抓出刀具磨損與換刀等特殊原因）", fontsize=12, fontweight='bold')
ax1.set_ylabel("軸心外徑平均值 (mm)", fontsize=10)
ax1.set_ylim(9.97, 10.04)
ax1.grid(True, linestyle='--', alpha=0.5)
ax1.legend(loc='lower left', ncol=5, fontsize=8)

# --- 子圖 2：R 管制圖 ---
ax2.plot(subgroups, df['R'], marker='s', color='#ff7f0e', linewidth=1.5, label=r'子組全距 ($R$)')
ax2.axhline(UCL_R, color='red', linestyle='--', label=f'UCL = {UCL_R:.3f}')
ax2.axhline(CL_R, color='green', linestyle='-', label=f'CL = {CL_R:.3f}')
ax2.axhline(LCL_R, color='red', linestyle='--', label=f'LCL = {LCL_R:.3f}')

ax2.set_title(r"$R$ 管制圖（反映「子組內變異」：條件一致時僅含純隨機噪聲，製程散佈穩定）", fontsize=12, fontweight='bold')
ax2.set_xlabel("抽樣子組編號（每 2 小時抽取連續 5 支）", fontsize=10)
ax2.set_ylabel("全距 R (mm)", fontsize=10)
ax2.set_xticks(subgroups)
ax2.set_xticklabels([f"G{k}\n({t})" for k, t in zip(subgroups, timestamps)], fontsize=8)
ax2.grid(True, linestyle='--', alpha=0.5)
ax2.legend(loc='upper right', ncol=4, fontsize=8)

plt.tight_layout()
plt.show()